<a href="https://colab.research.google.com/github/mandevautospa/GlobalTerror_NeuralNetwork/blob/main/gtd_nn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import pandas as pd
import kagglehub
import torch
import torch.nn as nn
import requests
import os

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics.pairwise import haversine_distances

path = kagglehub.dataset_download("START-UMD/gtd")
print(tf.version) #ensure version 2.x

Using Colab cache for faster access to the 'gtd' dataset.
<module 'tensorflow._api.v2.version' from '/usr/local/lib/python3.12/dist-packages/tensorflow/_api/v2/version/__init__.py'>


In [ ]:
for f in os.listdir(path):
  if f.endswith(".csv"):
    csv_path = os.path.join(path, f)
df = pd.read_csv(csv_path, encoding ="latin1", low_memory=False)
df = df.replace({-99:None})
df.head()

features = ["iyear", "imonth", "iday",
            "country", "region",
            "latitude", "longitude",
            "nkill", "nwound", "gname"]

targets = ["attacktype1", "success", "gname", "nkill", "nwound"]
all_cols = list(dict.fromkeys(features + targets))
df = df[all_cols].dropna()

encoders = {}
cat_cols = ["country", "region", "gname", "attacktype1"]

for col in cat_cols:
  le = LabelEncoder()
  df[col] = le.fit_transform(df[col].astype(str))
  encoders[col] = le

# All numeric feature columns (scaling deferred to after train/val split)
num_cols = ["iyear", "imonth", "iday", "latitude", "longitude", "nkill", "nwound"]


In [ ]:
import os

# iso3_map: map ACLED country names to ISO 3166-1 alpha-3 codes
iso3_map = {}

ACLED_URL = "https://api.acleddata.com/acled/read"

# Retrieve ACLED credentials from Colab userdata or environment variables
try:
    from google.colab import userdata
    acled_email = userdata.get("ACLED_EMAIL")
    acled_key   = userdata.get("ACLED_KEY")
except Exception:
    acled_email = os.environ.get("ACLED_EMAIL", "")
    acled_key   = os.environ.get("ACLED_KEY", "")

params = {"limit": 1000, "year": 2026, "email": acled_email, "key": acled_key}

try:
    r = requests.get(ACLED_URL, params=params, timeout=30)
    r.raise_for_status()
    acled = pd.DataFrame(r.json()['data'])
    acled['event_date_utc'] = pd.to_datetime(acled['event_date'], utc=True)
    acled['country_iso3'] = acled['country'].map(iso3_map)
    acled['reported_by_acled'] = 1
except Exception as e:
    print(f"ACLED data unavailable: {e}")
    acled = pd.DataFrame(columns=['event_date', 'event_date_utc', 'country', 'country_iso3', 'reported_by_acled'])

In [ ]:
print(df.columns.tolist())

['iyear', 'imonth', 'iday', 'country', 'region', 'latitude', 'longitude', 'nkill', 'nwound', 'gname', 'attacktype1', 'success']


In [ ]:
# Temporal train/validation split: train on older data, validate on newer data
# Scaler is fit on training data only to prevent data leakage
# split_year can be adjusted: earlier years give more training data,
# later years give a larger/more-recent validation set
split_year = 2015
df_train = df[df["iyear"] <= split_year].copy()
df_val   = df[df["iyear"] > split_year].copy()

scaler = StandardScaler()
df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
df_val[num_cols]   = scaler.transform(df_val[num_cols])


In [ ]:
X_train = torch.tensor(df_train.drop(columns=targets).values, dtype=torch.float32)
X_val   = torch.tensor(df_val.drop(columns=targets).values, dtype=torch.float32)

y_attacktype_train = torch.tensor(df_train["attacktype1"].values, dtype=torch.long)
y_success_train    = torch.tensor(df_train["success"].values, dtype=torch.float32)
y_group_train      = torch.tensor(df_train["gname"].values, dtype=torch.long)
y_casualties_train = torch.tensor(df_train[["nkill", "nwound"]].values, dtype=torch.float32)

y_attacktype_val   = torch.tensor(df_val["attacktype1"].values, dtype=torch.long)
y_success_val      = torch.tensor(df_val["success"].values, dtype=torch.float32)
y_group_val        = torch.tensor(df_val["gname"].values, dtype=torch.long)
y_casualties_val   = torch.tensor(df_val[["nkill", "nwound"]].values, dtype=torch.float32)


In [ ]:
# ── 1. Unified multi-output TensorDataset & DataLoaders ───────────────────
train_dataset = TensorDataset(
    X_train,
    y_attacktype_train,
    y_success_train,
    y_group_train,
    y_casualties_train
)

val_dataset = TensorDataset(
    X_val,
    y_attacktype_val,
    y_success_val,
    y_group_val,
    y_casualties_val
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


In [ ]:
# ── 2. Multi-task neural network ─────────────────────────────────────────
class MultiTaskGTD(nn.Module):
    """Shared encoder with four task-specific output heads.

    Heads
    -----
    attack_type : multi-class  (CrossEntropyLoss)
    success     : binary       (BCEWithLogitsLoss)
    group       : multi-class  (CrossEntropyLoss)
    casualties  : 2-dim regression for [nkill, nwound] (MSELoss)
    """

    def __init__(self, input_dim, num_attacktypes, num_groups, hidden_dim=256, dropout=0.3):
        super().__init__()

        # Shared encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )

        # Task heads
        self.head_attacktype = nn.Linear(hidden_dim, num_attacktypes)
        self.head_success    = nn.Linear(hidden_dim, 1)
        self.head_group      = nn.Linear(hidden_dim, num_groups)
        self.head_casualties = nn.Linear(hidden_dim, 2)  # outputs [nkill, nwound]

    def forward(self, x):
        shared = self.encoder(x)
        attacktype_out = self.head_attacktype(shared)
        success_out    = self.head_success(shared).squeeze(1)  # (B,)
        group_out      = self.head_group(shared)
        casualties_out = self.head_casualties(shared)
        return attacktype_out, success_out, group_out, casualties_out


# Infer dimensions from tensors
# Use torch.unique to handle non-contiguous or missing label values safely
input_dim       = X_train.shape[1]
num_attacktypes = len(torch.unique(y_attacktype_train))
num_groups      = len(torch.unique(y_group_train))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = MultiTaskGTD(input_dim, num_attacktypes, num_groups).to(device)
print(model)
print(f"\ninput_dim={input_dim}, num_attacktypes={num_attacktypes}, num_groups={num_groups}")


In [ ]:
# ── 3 & 4. Multi-task training + validation loop ─────────────────────────
BINARY_THRESHOLD = 0.0  # decision boundary for success head logits

ce_loss  = nn.CrossEntropyLoss()
bce_loss = nn.BCEWithLogitsLoss()
mse_loss = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
NUM_EPOCHS = 20

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Training ──────────────────────────────────────────────────────────
    model.train()
    total_loss = total_att = total_suc = total_grp = total_cas = 0.0

    for X_b, y_att, y_suc, y_grp, y_cas in train_loader:
        X_b   = X_b.to(device)
        y_att = y_att.to(device)
        y_suc = y_suc.to(device)
        y_grp = y_grp.to(device)
        y_cas = y_cas.to(device)

        att_out, suc_out, grp_out, cas_out = model(X_b)

        loss_attack     = ce_loss(att_out, y_att)
        loss_success    = bce_loss(suc_out, y_suc.float())
        loss_group      = ce_loss(grp_out, y_grp)
        loss_casualties = mse_loss(cas_out, y_cas.float())

        loss = loss_attack + loss_success + loss_group + loss_casualties

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_att  += loss_attack.item()
        total_suc  += loss_success.item()
        total_grp  += loss_group.item()
        total_cas  += loss_casualties.item()

    n = len(train_loader)
    train_log = (
        f"Epoch {epoch:>2}/{NUM_EPOCHS} | "
        f"Train total={total_loss/n:.4f}  "
        f"att={total_att/n:.4f}  "
        f"suc={total_suc/n:.4f}  "
        f"grp={total_grp/n:.4f}  "
        f"cas={total_cas/n:.4f}"
    )

    # ── Validation ────────────────────────────────────────────────────────
    model.eval()
    v_loss = v_att = v_suc = v_grp = v_cas = 0.0
    correct_att = correct_suc = correct_grp = 0
    total_samples = 0

    with torch.no_grad():
        for X_b, y_att, y_suc, y_grp, y_cas in val_loader:
            X_b   = X_b.to(device)
            y_att = y_att.to(device)
            y_suc = y_suc.to(device)
            y_grp = y_grp.to(device)
            y_cas = y_cas.to(device)

            att_out, suc_out, grp_out, cas_out = model(X_b)

            v_att += ce_loss(att_out, y_att).item()
            v_suc += bce_loss(suc_out, y_suc.float()).item()
            v_grp += ce_loss(grp_out, y_grp).item()
            v_cas += mse_loss(cas_out, y_cas.float()).item()

            batch_size = X_b.size(0)
            total_samples += batch_size

            correct_att += (att_out.argmax(1) == y_att).sum().item()
            correct_suc += ((suc_out > BINARY_THRESHOLD).long() == y_suc.long()).sum().item()
            correct_grp += (grp_out.argmax(1) == y_grp).sum().item()

    m = len(val_loader)
    acc_att = correct_att / total_samples
    acc_suc = correct_suc / total_samples
    acc_grp = correct_grp / total_samples
    val_total = v_att + v_suc + v_grp + v_cas

    val_log = (
        f"          | "
        f"  Val total={val_total/m:.4f}  "
        f"att={v_att/m:.4f}(acc={acc_att:.3f})  "
        f"suc={v_suc/m:.4f}(acc={acc_suc:.3f})  "
        f"grp={v_grp/m:.4f}(acc={acc_grp:.3f})  "
        f"cas_mse={v_cas/m:.4f}"
    )

    print(train_log)
    print(val_log)


In [ ]:
print(df.dtypes)


iyear          float64
imonth         float64
iday           float64
country          int64
region           int64
latitude       float64
longitude      float64
nkill          float64
nwound         float64
gname            int64
attacktype1      int64
success          int64
dtype: object
